# EURUSD M1 — Cleaning & Parquet Conversion

**Source**: MetaTrader 5 export (View → Symbols → Bars → Request)  
**Pair**: EURUSD | **Timeframe**: M1 | **Date range**: 2000-01-03 → 2025-05-23  
**Pull date**: 2026-05-31  
**Format**: Tab-separated, no header, 9 columns

In [ ]:
# Run this notebook from the project root, or set PROJECT_ROOT below
import os
PROJECT_ROOT = r"C:\Projects\My\Re-Diggn"
os.chdir(PROJECT_ROOT)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# The full script is in scripts/clean_eurusd_m1.py
# Uncomment to run it inline:
# %run scripts/clean_eurusd_m1.py

# Or run from terminal:
#   uv run python scripts/clean_eurusd_m1.py

In [ ]:
import glob
import time
import numpy as np
import pandas as pd

RAW_DIR  = os.path.join(PROJECT_ROOT, "data", "raw")
OUT_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "eurusd_m1.parquet")
np.random.seed(42)

COL_NAMES = ["date", "time", "open", "high", "low", "close",
             "tick_volume", "volume", "spread"]

csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "EURUSD_i_M1_*.csv")))
print(f"Found {len(csv_files)} CSV files")
for f in csv_files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1024**2:.1f} MB)")

In [ ]:
# ── Load all CSVs ────────────────────────────────────────────────────────────
t0 = time.time()
frames = []
total_raw = 0

for f in csv_files:
    df_part = pd.read_csv(
        f, sep="\t", header=None, names=COL_NAMES,
        dtype={
            "date": str, "time": str,
            "open": "float32", "high": "float32",
            "low": "float32",  "close": "float32",
            "tick_volume": "int32", "volume": "int32", "spread": "int16",
        },
    )
    frames.append(df_part)
    total_raw += len(df_part)
    print(f"  {os.path.basename(f)}: {len(df_part):,} rows")

df = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df):,} rows  ({time.time()-t0:.1f}s)")

In [ ]:
# ── Parse timestamps → UTC ───────────────────────────────────────────────────
df["timestamp"] = pd.to_datetime(
    df["date"] + " " + df["time"],
    format="%Y.%m.%d %H:%M:%S",
    utc=True,
)
df = df.drop(columns=["date", "time"]).set_index("timestamp")
print(f"Index dtype: {df.index.dtype}")
print(f"Range: {df.index.min()} → {df.index.max()}")

In [ ]:
# ── Data Quality Checks ──────────────────────────────────────────────────────
issues = {}

# Duplicates
n_dupes = int(df.index.duplicated(keep="last").sum())
issues["duplicate_timestamps"] = n_dupes
print(f"Duplicates           : {n_dupes:,}")
if n_dupes > 0:
    df = df[~df.index.duplicated(keep="last")]

# Out-of-order
if not df.index.is_monotonic_increasing:
    n_ooo = int((df.index.to_series().diff().dt.total_seconds() < 0).sum())
    issues["out_of_order_rows"] = n_ooo
    print(f"Out-of-order rows    : {n_ooo:,} → sorting")
    df = df.sort_index()
else:
    issues["out_of_order_rows"] = 0
    print("Out-of-order rows    : 0  (index already sorted)")

# Zero / negative prices
mask_nonpos = (df[["open","high","low","close"]] <= 0).any(axis=1)
n_nonpos = int(mask_nonpos.sum())
issues["zero_or_negative_prices"] = n_nonpos
print(f"Zero/neg prices      : {n_nonpos:,}")
if n_nonpos > 0:
    df = df[~mask_nonpos]

# OHLC hard: high < low
bad = df["high"] < df["low"]
n_bad = int(bad.sum())
issues["ohlc_high_lt_low_dropped"] = n_bad
print(f"OHLC high<low        : {n_bad:,} → dropping")
if n_bad > 0:
    df = df[~bad]

# OHLC soft: flagged only
soft = (df["high"] < df[["open","close"]].max(axis=1)) | \
       (df["low"]  > df[["open","close"]].min(axis=1))
n_soft = int(soft.sum())
issues["ohlc_soft_violations_flagged"] = n_soft
print(f"OHLC soft violations : {n_soft:,}  (flagged, kept)")

# Weekend bars
mask_wknd = df.index.dayofweek >= 5
n_wknd = int(mask_wknd.sum())
issues["weekend_bars_dropped"] = n_wknd
print(f"Weekend bars         : {n_wknd:,} → dropping")
df = df[~mask_wknd]

# Zero tick_volume
n_ztv = int((df["tick_volume"] == 0).sum())
issues["zero_tick_volume_flagged"] = n_ztv
print(f"Zero tick_volume     : {n_ztv:,}  (flagged, kept)")

In [ ]:
# ── Gap Analysis ─────────────────────────────────────────────────────────────
gaps_s = df.index.to_series().diff().dt.total_seconds().dropna()
print(f"Median gap: {gaps_s.median():.0f}s  (expected 60s)")
long_gaps = gaps_s[gaps_s > 3600]
print(f"Gaps > 60 min: {len(long_gaps):,}")

gap_end_pos   = df.index.get_indexer(long_gaps.index)
gap_start_ts  = df.index[gap_end_pos - 1]
gap_df = pd.DataFrame({
    "gap_start": gap_start_ts,
    "gap_end":   long_gaps.index,
    "gap_hours": long_gaps.values / 3600,
    "is_weekend": gap_start_ts.dayofweek >= 4,
})
print(f"  Weekend gaps:    {gap_df['is_weekend'].sum():,}")
print(f"  Intra-week gaps: {(~gap_df['is_weekend']).sum():,}")

intraweek = gap_df[~gap_df["is_weekend"]].sort_values("gap_hours", ascending=False)
if len(intraweek):
    print("\nTop intra-week gaps (hours):")
    print(intraweek.head(15).to_string(index=False))

In [ ]:
# ── Final dtypes & Write Parquet ─────────────────────────────────────────────
df["open"]        = df["open"].astype("float32")
df["high"]        = df["high"].astype("float32")
df["low"]         = df["low"].astype("float32")
df["close"]       = df["close"].astype("float32")
df["tick_volume"] = df["tick_volume"].astype("int32")
df["volume"]      = df["volume"].astype("int32")
df["spread"]      = df["spread"].astype("int16")

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_parquet(OUT_PATH, engine="pyarrow", compression="snappy", index=True)
fsize = os.path.getsize(OUT_PATH) / 1024**2
mem   = df.memory_usage(deep=True).sum() / 1024**2

print(f"Rows after cleaning  : {len(df):,}")
print(f"Date range           : {df.index.min()} → {df.index.max()}")
print(f"DataFrame memory     : {mem:.1f} MB")
print(f"Parquet file (snappy): {fsize:.2f} MB")
print(f"Output path          : {OUT_PATH}")

In [ ]:
# ── Verification read-back ───────────────────────────────────────────────────
df_check = pd.read_parquet(OUT_PATH)
df_check.info(memory_usage="deep")
df_check.describe()